# KRONOS2 zero-shot cell embeddings vs KRONOS (v1) and the cell-table models (Jackson, Danenberg)

**Why KRONOS2.** KRONOS2 (MahmoodLab, 2026; marker-aware DINOv2 ViT-B/16, 768-d) is the successor of KRONOS. Its
pretraining set, SPM-56M (model card figure), has 56 M single-marker patches from 944 images and 8 *fluorescence* platforms:
CODEX, CellDive, COMET, MxIF, IBEX, CosMx, Orion and Vectra. It has no IMC and no MIBI, and breast is 0.06% of it.
So, like KRONOS, it saw neither of our cohorts.

**Design: one change from `kronos-zeroshot`.** Cells, labels, splits, table features, cell patches, masking, normalisation
calibration, scorer and patient bootstrap are copied from `kronos-zeroshot`. Only the encoder changes. The KRONOS (v1)
embeddings from that run's output are scored again in the same splits, so every v1-vs-v2 difference is paired.

**Inputs.** The same 13 channels as v1:
- Names from the KRONOS2 vocabulary: DAPI = mean(DNA1, DNA2), HISTONEH3, HER2, CD20, CD3, CD45, CD68, ERA (ERα; in the KRONOS2
  vocabulary, unlike v1), KI67, CYTOKERATIN (panCK), A-SMA.
- CK5 and CK8/18 are not in the vocabulary. They are registered as novel markers with KRONOS2's own route,
  `register_additional_markers`: a BioLinkBERT text embedding of the marker description, with compartment "cytoplasm" and family
  "epithelial_keratin" (as CK20).

**Cell patches (as v1).**
- A 32 × 32 px window around the mask centroid, upsampled ×2 to 64 × 64 px, so 4 × 4 ViT tokens per marker.
- Each channel is scaled to [0, 1] by its cohort-wide 99.9th percentile, then standardised.
- Pixels outside the cell's mask are set to 0.

**Embedding per cell (as v1):** the CLS token (768), plus the first 256 principal components of the per-marker features. The
per-marker features are the mean of each marker's 16 patch tokens, flattened (13 × 768). The PCA is fitted on calibration images.

**Normalisation choice (as v1, decided on training data only):** KRONOS2's per-marker mean / SD (novel markers: our pixel
statistics) vs our cohort's own pixel mean / SD. Both are scored by a coarse linear probe, fitted on 30 Jackson replicate-0
training images and evaluated on the Jackson replicate-0 validation images. The better one is used.

**Arms** (LightGBM, logistic regression, scorer and bootstrap as `cell-type-graph` / `kronos-zeroshot`):

| Arm | Features |
|---|---|
| Table | 14 shared channels + log area |
| Table + nbr | + kNN-10 neighbour features |
| KRONOS linear / KRONOS2 linear | the encoder's embedding alone (v1: 640-d; v2: 1024-d), logistic regression |
| Table + K32 / Table + K2-32 | Table + first 32 per-marker PCs of KRONOS / KRONOS2 |
| Table + nbr + K32 / Table + nbr + K2-32 | Table + nbr + the same 32 PCs |

Tasks: coarse, trained on Jackson, Danenberg or both, tested on each cohort. Fine, within each cohort, uses only the nbr arms
and the two linear probes. **Replicates 0, 1 and 2**; `kronos-zeroshot` ran replicate 0 only.

**Gate for adaptation (pre-registered before the full run; `future_plan.md` §0.3 item 5).** Applied to each encoder separately:
- **(a)** its linear probe is within 0.05 macro-F1 of Table + nbr on coarse J→J or D→D;
- **(b)** Table + nbr + its 32 PCs beats Table + nbr by ≥ 0.01 macro-F1 (paired CI > 0) on any coarse task.

An encoder passes a replicate if (a) or (b) holds. It passes the gate only if it passes **every replicate run**. The
replicate-0-only verdict (the v1 rule) is also reported, for continuity.

**Head-to-head:** paired differences KRONOS2 − KRONOS for the linear probes and the +32-PC arms, per replicate.

The KRONOS2 embeddings are saved for the deferred imputation and H3K27me3 probes. Those are scored only with the G-R5 leak
controls.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc, traceback, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
pip = [sys.executable, "-m", "pip", "install", "-q"]
if subprocess.run(pip + ["--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run(pip + ["--only-binary=:all:", "imagecodecs"], check=True)
try:
    import omegaconf
except ImportError:
    subprocess.run(pip + ["omegaconf"], check=True)
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1" or os.path.exists(f"{IN}/SMOKE")
SEED = 20261013                                   # same seed and split rules as cell-type-graph / kronos-zeroshot
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0                                         # table features exactly as in cell-type-graph
N_FIT = 20_000 if SMOKE else 300_000
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
REPS = [int(r) for r in os.environ.get("REPS", "0,1,2").split(",")]
SMOKE_IMAGES = 24
CROP, UP = 32, 2                                  # 32 px window at 1 µm/px, upsampled to a 64 px cell patch (as v1)
Q_NORM = 99.9
BATCH = 256
N_PCA, N_K = 256, 32
N_CAL_FIT, N_CAL_PCA = (8, 8) if SMOKE else (30, 40)
BUDGET_MIN = 660.0
dev = "cuda" if torch.cuda.is_available() else "cpu"
T0 = time.time()
FAILED = []

NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = ["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
           "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"]
FEATS = NUC + MARKERS
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
IMG_CHAN = {"Danenberg": D_CHAN, "Jackson": J_CHAN}
# KRONOS2 marker name -> source image channels (averaged); same 13 channels and order as v1
KCH = [("DAPI", ["dna1", "dna2"]), ("HISTONEH3", ["histone_h3"]), ("HER2", ["c_erb_b_2_her2"]), ("CD20", ["cd20"]),
       ("CD3", ["cd3"]), ("CD45", ["cd45"]), ("CD68", ["cd68"]), ("CK5", ["cytokeratin_5"]),
       ("CK8_18", ["cytokeratin_8_18"]), ("ERA", ["estrogen_receptor_alpha"]), ("KI67", ["ki_67"]),
       ("CYTOKERATIN", ["pan_cytokeratin"]), ("A-SMA", ["sma"])]
NOVEL = {"CK5": "Cytokeratin 5 (KRT5)", "CK8_18": "Cytokeratin 8/18 (KRT8/KRT18)"}   # not in the KRONOS2 vocabulary
NK = len(KCH)

COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

print("SMOKE" if SMOKE else "FULL", "| device", dev, "| GPUs", torch.cuda.device_count(), "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| reps", REPS)

## KRONOS2 model and marker vocabulary

In [ ]:
# The dataset holds the checkpoint and the Hugging Face repo's code as kronos2_code.tar. The code goes to a writable folder:
# registering novel markers caches BioLinkBERT there.
K2DIR = "/tmp/kronos2"
os.makedirs(K2DIR, exist_ok=True)
hit = g("modeling_kronos2.py")
if hit:                                           # Kaggle unpacked the archive
    shutil.copytree(os.path.dirname(hit[0]), K2DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pth"))
else:
    shutil.unpack_archive(g("kronos2_code.tar")[0], K2DIR)
assert os.path.isdir(f"{K2DIR}/dinov2"), os.listdir(K2DIR)
if not os.path.exists(f"{K2DIR}/kronos2_vitb16_teacher.pth"):
    os.symlink(g("kronos2_vitb16_teacher.pth")[0], f"{K2DIR}/kronos2_vitb16_teacher.pth")
sys.path.insert(0, os.path.dirname(K2DIR))
from kronos2.modeling_kronos2 import Kronos2Model
from kronos2.marker_utils import marker_match_key, build_marker_index, MARKER_MEAN, MARKER_STD
k2 = Kronos2Model.from_pretrained(K2DIR, device=dev)
mb = k2.backbone.marker_builder

# CK5 and CK8/18: KRONOS2's novel-marker route (BioLinkBERT text embedding; existing compartment / family, as CK20)
base_csv = pd.read_csv(f"{K2DIR}/marker_metadata.csv")
tmpl = base_csv[base_csv.marker_name == "CK20"].iloc[0]
pd.DataFrame([dict(marker_name=n, compartment=tmpl.compartment, family=tmpl.family, compartment_desc=tmpl.compartment_desc,
                   family_desc=tmpl.family_desc, marker_full_name=full, mean=MARKER_MEAN, std=MARKER_STD, pretraining="no")
              for n, full in NOVEL.items()]).to_csv("/tmp/kronos2_novel_markers.csv", index=False)
k2.register_additional_markers("/tmp/kronos2_novel_markers.csv")

idx = build_marker_index(k2._marker_stats)
meta = base_csv.assign(key=base_csv.marker_name.map(marker_match_key)).drop_duplicates("key").set_index("key")
MU_K, SD_K, rows = [], [], []
for name, _ in KCH:
    pos = int(mb._fuzzy_match([name])[0])        # raises if the name does not resolve
    known = name not in NOVEL
    if known:
        r = meta.loc[marker_match_key(name)]
        assert r.pretraining == "yes", name
        MU_K.append(float(r["mean"])); SD_K.append(float(r["std"]))
    else:
        MU_K.append(np.nan); SD_K.append(np.nan)
    rows.append(dict(channel=name, vocab_index=pos, in_pretraining=known, kronos2_mean=MU_K[-1], kronos2_sd=SD_K[-1]))
MU_K, SD_K = np.array(MU_K, np.float32), np.array(SD_K, np.float32)
print(pd.DataFrame(rows).to_string())

# sanity check of the novel markers: nearest vocabulary markers by text embedding
E_txt = F.normalize(mb.precomputedMarkerTextEmb.float(), dim=1)
names_by_pos = {v: k for k, v in mb.markerNameToIndex.items()}
for n in NOVEL:
    p = mb.markerNameToIndex[n.lower()]
    sim = (E_txt @ E_txt[p]).cpu().numpy()
    top = [names_by_pos.get(j, str(j)) for j in np.argsort(-sim)[1:6]]
    print(f"{n}: nearest vocabulary markers by text embedding {top}")

# All cells share one marker list, so the marker embedding (eval mode: deterministic) is computed once here and stored as a
# buffer, instead of once per cell. DataParallel replicas carry buffers but expose no parameters(), which the builder's
# forward needs.
import dinov2.models.markers.builder as k2builder
_mb_forward = k2builder.MarkerEmbeddingBuilder.forward
K2_NAMES = [n for n, _ in KCH]
with torch.no_grad():
    mb.register_buffer("cached_marker_emb", _mb_forward(mb.eval(), [K2_NAMES], 1).detach().float())   # 1, 13, 768
def _mb_forward_cached(self, marker_names, B, present_mask=None):
    if present_mask is None and hasattr(self, "cached_marker_emb") and all(m == K2_NAMES for m in marker_names):
        return self.cached_marker_emb.expand(B, -1, -1)
    return _mb_forward(self, marker_names, B, present_mask)
k2builder.MarkerEmbeddingBuilder.forward = _mb_forward_cached

class K2Net(nn.Module):
    """CLS token and per-marker mean patch tokens (13 × 768) for a batch of 13-channel cell patches."""
    def __init__(self, backbone, names):
        super().__init__()
        self.backbone, self.names = backbone, list(names)
    def forward(self, x):
        o = self.backbone.forward_features(x, masks=None, marker_names=[self.names] * x.shape[0])
        t = o["x_norm_patchtokens"]                                                  # B, 13 * tokens, 768 (marker-major)
        return o["x_norm_clstoken"], t.reshape(t.shape[0], len(self.names), -1, t.shape[-1]).mean(2)

net = K2Net(k2.backbone, K2_NAMES).eval().to(dev)
NET = nn.DataParallel(net) if torch.cuda.device_count() > 1 else net
D_EMB = 768
K2_INFO = dict(repo="MahmoodLab/KRONOS2", arch="ViT-B/16", embed_dim=D_EMB,
               checkpoint_sha256="afa261ef8b63bb0acea62612d7594ac11fbdb89fbdd23c513f046a034725d2b4",
               novel_markers=list(NOVEL))
tick(f"KRONOS2 loaded, embedding dim {D_EMB}")

## Cells, labels, splits and table features (as in `cell-type-graph`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    names = pq.read_schema(p).names
    if "object_number" not in names or "published__h3k27me3" in names:
        continue
    key = "Danenberg" if "metabric_id" in names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p))
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))

IMGP, MASKP = {"Danenberg": {}, "Jackson": {}}, {"Danenberg": {}, "Jackson": {}}
for p in g("*.tiff"):
    q = p.replace("\\", "/")
    if "keren" in q.lower():
        continue
    name = os.path.splitext(os.path.basename(p))[0]
    co = "Danenberg" if name.startswith("MB_") else "Jackson"
    (MASKP if "/masks/" in q else IMGP)[co][name] = p
print({c: (len(IMGP[c]), len(MASKP[c])) for c in IMGP})

def load(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "Danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    keep = n_per.index[n_per >= MIN_CELLS]
    keep = keep[keep.isin(list(set(IMGP[key]) & set(MASKP[key])))]
    if SMOKE:
        keep = np.sort(np.random.default_rng(SEED).choice(np.asarray(keep), SMOKE_IMAGES, replace=False))
    cells = cells[cells.image_name.isin(keep)].sort_values(["image_name", "object_number"]).reset_index(drop=True)
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    CON = sp.coo_matrix((np.ones(int((ca != cb).sum())), (ca[ca != cb], cb[ca != cb])), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float64).tocsr()
    cols = [D_TABLE.get(c, c) for c in FEATS] if key == "Danenberg" else FEATS
    X = np.c_[np.arcsinh(cells[cols].values.astype(np.float64) / COF),
              np.log(np.maximum(cells["area"].values, 1))].astype(np.float32)
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    return dict(key=key, n=n, starts=starts, n_img=len(img_names), img_names=np.asarray(img_names),
                obj=cells.object_number.values.astype(np.int64), patient=cells[pid].astype(str).values,
                xy=cells[["x", "y"]].values.astype(np.float64), CON=CON, X=X, fine=fine,
                y={"fine": y_fine, "coarse": coarse_of[y_fine]})

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

def knn10(C, k=10):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        kk = min(k, b - a - 1)
        _, nb = cKDTree(C["xy"][a:b]).query(C["xy"][a:b], k=kk + 1)
        rows.append(np.repeat(np.arange(a, b), kk)); cols.append(nb[:, 1:].ravel() + a)
    A = sp.coo_matrix((np.ones(sum(map(len, rows)), np.float32), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def r30_count(C):
    out = np.zeros(C["n"], np.int64)
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        out[a:b] = cKDTree(C["xy"][a:b]).query_ball_point(C["xy"][a:b], 30.0, return_length=True) - 1
    return out

def nbr_features(C, A):
    X = C["XS"]["cohort"]
    deg = np.diff(A.indptr).astype(np.float32)[:, None]
    mean = (A @ X) / deg
    std = np.sqrt(np.maximum((A @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A.indices], A.indptr[:-1], axis=0)
    hop2 = (A @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    return np.c_[X, mean, mx, std, hop2, np.log1p(r30_count(C)), np.log1p(np.diff(C["CON"].indptr)), dist / 30.0].astype(np.float32)

for key, C in CO.items():
    C["XS"] = {"cohort": np.nan_to_num(harm(C["X"]))}
    C["XS"]["nbr"] = nbr_features(C, knn10(C))
    print(f"{key}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients, {len(C['fine'])} fine labels")
tick("tables, labels and neighbour features")

## Splits (replicate 0 decides the calibration)

In [ ]:
D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def split_images(C, cell_split):
    s = cell_split[C["starts"][:-1]]
    return {k: np.flatnonzero(s == k) for k in ("train", "val", "test")}

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    IMG = {"Jackson": split_images(CJ, J_SPLIT), "Danenberg": split_images(CD, D_SPLIT)}
    IMG["Both"] = {s: np.r_[IMG["Jackson"][s], IMG["Danenberg"][s] + CJ["n_img"]] for s in ("train", "val")}
    return IMG

def cells_of(C, imgs):
    imgs = np.asarray(imgs, np.int64)
    if not len(imgs):
        return np.zeros(0, np.int64)
    return np.concatenate([np.arange(C["starts"][i], C["starts"][i + 1]) for i in imgs])

IMG0 = make_splits(0)
for c in ("Jackson", "Danenberg"):
    print(c, {s: len(v) for s, v in IMG0[c].items()}, "images")

## Pixel statistics and the KRONOS2 embedding pass

In [ ]:
def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im

def load_image(key, i):
    """Raw 13-channel stack (H, W, 13), the mask, and the centroids of this image's table cells."""
    C = CO[key]
    s = C["img_names"][i]
    im = read_hwc(IMGP[key][s], len(IMG_CHAN[key]))
    X = np.stack([im[..., [IMG_CHAN[key].index(c) for c in srcs]].astype(np.float32).mean(-1) for _, srcs in KCH], -1)
    mask = tifffile.imread(MASKP[key][s]).astype(np.int64)
    assert mask.shape == X.shape[:2], (s, mask.shape, X.shape)
    obj = C["obj"][C["starts"][i]:C["starts"][i + 1]]
    flat = mask.ravel()
    k = max(int(flat.max()), int(obj.max())) + 1
    cnt = np.bincount(flat, minlength=k)
    yy, xx = np.divmod(np.arange(flat.size), mask.shape[1])
    cy = np.bincount(flat, yy, k)[obj] / np.maximum(cnt[obj], 1)
    cx = np.bincount(flat, xx, k)[obj] / np.maximum(cnt[obj], 1)
    return X, mask, obj, np.round(cy).astype(np.int64), np.round(cx).astype(np.int64), cnt[obj] > 0

# cohort-wide per-channel 99.9th percentile (scale to [0, 1]) and own pixel mean / SD inside cells, from training images
STATS = {}
for key in ("Jackson", "Danenberg"):
    rs = np.random.default_rng(SEED + 5)
    pick = rs.choice(IMG0[key]["train"], min(40, len(IMG0[key]["train"])), replace=False)
    with ThreadPoolExecutor(4) as ex:
        ims = list(ex.map(lambda i: load_image(key, i)[:2], pick))
    q = np.percentile(np.concatenate([x.reshape(-1, NK)[::7] for x, _ in ims]), Q_NORM, axis=0)
    q = np.maximum(q, 1e-6)
    v = np.concatenate([np.clip(x.reshape(-1, NK)[m.ravel() > 0] / q, 0, 1) for x, m in ims])
    STATS[key] = dict(q=q.astype(np.float32), mu=v.mean(0).astype(np.float32), sd=np.maximum(v.std(0), 1e-4).astype(np.float32))
    del ims, v
    print(key, pd.DataFrame({k_: STATS[key][k_] for k_ in ("q", "mu", "sd")}, index=[n for n, _ in KCH]).round(4).T.to_string())

def norm_params(key, variant):
    st = STATS[key]
    if variant == "kronos2":
        mu = np.where(np.isnan(MU_K), st["mu"], MU_K).astype(np.float32)
        sd = np.where(np.isnan(SD_K), st["sd"], SD_K).astype(np.float32)
    else:
        mu, sd = st["mu"], st["sd"]
    t = lambda a: torch.tensor(a, device=dev)
    return t(st["q"]), t(mu), t(sd)

@torch.no_grad()
def embed_image(key, i, variant, data=None):
    X, mask, obj, cy, cx, has = data if data is not None else load_image(key, i)
    q, mu, sd = norm_params(key, variant)
    z = ((torch.from_numpy(X).to(dev) / q).clamp(0, 1) - mu) / sd                 # H, W, 13
    pad = CROP // 2
    zp = F.pad(z.permute(2, 0, 1), (pad, pad, pad, pad))                           # 13, H+2p, W+2p
    lp = F.pad(torch.from_numpy(mask).to(dev), (pad, pad, pad, pad))
    off = torch.arange(-pad, pad, device=dev)
    cyt, cxt, objt = (torch.from_numpy(a).to(dev) for a in (cy, cx, obj))
    cls_out = np.zeros((len(obj), D_EMB), np.float32)
    mrk_out = np.zeros((len(obj), NK * D_EMB), np.float16)
    for s in range(0, len(obj), BATCH):
        sl = slice(s, s + BATCH)
        r = cyt[sl, None] + pad + off[None, :]
        c = cxt[sl, None] + pad + off[None, :]
        patch = zp[:, r[:, :, None], c[:, None, :]].permute(1, 0, 2, 3)           # n, 13, 32, 32
        m = (lp[r[:, :, None], c[:, None, :]] == objt[sl, None, None]).float()     # n, 32, 32
        if UP > 1:
            patch = F.interpolate(patch, scale_factor=UP, mode="bilinear", align_corners=False)
            m = F.interpolate(m[:, None], scale_factor=UP, mode="nearest")[:, 0]
        patch = (patch * m[:, None]).contiguous()
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev == "cuda"):
            cls, mf = NET(patch)
        cls_out[s:s + BATCH] = cls.float().cpu().numpy()
        mrk_out[s:s + BATCH] = mf.reshape(mf.shape[0], -1).half().cpu().numpy()
    cls_out[~has] = np.nan
    return cls_out, mrk_out

def embed_images(key, idx, variant):
    """Embeds images idx of cohort key; image loading runs ahead in threads."""
    res = {}
    with ThreadPoolExecutor(3) as ex:
        futs = {i: ex.submit(load_image, key, i) for i in idx[:6]}
        for j, i in enumerate(idx):
            if j + 6 < len(idx):
                futs[idx[j + 6]] = ex.submit(load_image, key, idx[j + 6])
            res[i] = embed_image(key, i, variant, futs.pop(i).result())
    return res
tick("embedding helpers ready")

## Calibration on training data: normalisation variant, then the PCA

In [ ]:
def scale_fit(Z):
    mu, sd = np.nanmean(Z, 0), np.nanstd(Z, 0) + 1e-6
    return lambda A: np.nan_to_num((A - mu) / sd).astype(np.float32)

def fit_logistic(Z, y, ncls, wds, Zv, yv, seed):
    """Class-weighted multinomial logistic regression (GPU, full batch, LBFGS); weight decay chosen on (Zv, yv)."""
    torch.manual_seed(seed)
    Zt, yt = torch.tensor(Z, device=dev), torch.tensor(y, device=dev)
    w = torch.tensor(np.bincount(y, minlength=ncls), device=dev, dtype=torch.float32)
    w = torch.where(w > 0, w.sum() / (ncls * w.clamp(min=1)), torch.zeros_like(w))
    best = None
    for wd in wds:
        lin = nn.Linear(Z.shape[1], ncls).to(dev)
        opt = torch.optim.LBFGS(lin.parameters(), lr=1, max_iter=200 if not SMOKE else 50, history_size=20,
                                line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(lin(Zt), yt, weight=w) + wd * (lin.weight ** 2).sum()
            loss.backward()
            return loss
        opt.step(closure)
        with torch.no_grad():
            pv = lin(torch.tensor(Zv, device=dev)).argmax(1).cpu().numpy()
        f1 = macro_f1(yv, pv, ncls)
        if best is None or f1 > best[0]:
            best = (f1, wd, lin)
    return best

def macro_f1(y, p, ncls):
    cm = np.bincount(y * ncls + p, minlength=ncls * ncls).reshape(ncls, ncls).astype(float)
    tp, sup, prd = np.diag(cm), cm.sum(1), cm.sum(0)
    f1 = np.where(sup + prd > 0, 2 * tp / np.maximum(sup + prd, 1), 0)
    return float(f1[sup > 0].mean())

rs = np.random.default_rng(SEED + 6)
cal_fit = np.sort(rs.choice(IMG0["Jackson"]["train"], min(N_CAL_FIT, len(IMG0["Jackson"]["train"])), replace=False))
cal_val = np.sort(IMG0["Jackson"]["val"])
if len(cal_val) < 2:                              # smoke runs: too few validation images; use other training images
    cal_val = np.sort(np.setdiff1d(IMG0["Jackson"]["train"], cal_fit)[:4])
CAL = {}
for variant in ("kronos2", "own"):
    E = embed_images("Jackson", np.r_[cal_fit, cal_val], variant)
    def feats(idx):
        cl = np.vstack([E[i][0] for i in idx])
        mk = np.vstack([E[i][1].astype(np.float32).reshape(-1, NK, D_EMB).mean(1) for i in idx])
        rows = cells_of(CJ, idx)
        yv = CJ["y"]["coarse"][rows]
        ok = (yv >= 0) & np.isfinite(cl).all(1)
        return np.c_[cl, mk][ok], yv[ok]
    Zf, yf = feats(cal_fit)
    Zv, yv = feats(cal_val)
    sc = scale_fit(Zf)
    f1, wd, _ = fit_logistic(sc(Zf), yf, len(COARSE), [1e-4, 1e-3], sc(Zv), yv, SEED)
    CAL[variant] = dict(macro_f1_val=f1, weight_decay=wd, n_fit=int(len(yf)), n_val=int(len(yv)))
    print(variant, CAL[variant])
    del E
VARIANT = max(CAL, key=lambda v: CAL[v]["macro_f1_val"])
print("normalisation used:", VARIANT)

# PCA of the flattened per-marker features, fitted on training images of both cohorts (chosen variant)
mk_cal = []
for key in ("Jackson", "Danenberg"):
    pick = np.sort(np.random.default_rng(SEED + 8).choice(IMG0[key]["train"], min(N_CAL_PCA // 2, len(IMG0[key]["train"])),
                                                          replace=False))
    E = embed_images(key, pick, VARIANT)
    mk_cal.append(np.vstack([E[i][1] for i in pick]).astype(np.float32))
    del E
M = np.vstack(mk_cal)
M = M[np.isfinite(M).all(1)]
M = M[np.random.default_rng(SEED + 9).choice(len(M), min(60_000, len(M)), replace=False)]
Mt = torch.tensor(M, device=dev)
PCA_MU = Mt.mean(0, keepdim=True)
_, S_, V_ = torch.pca_lowrank(Mt - PCA_MU, q=N_PCA, center=False, niter=4)
PCA_V = V_[:, :N_PCA].contiguous()
expl = (S_[:N_PCA] ** 2).sum().item() / ((Mt - PCA_MU) ** 2).sum().item()
del Mt, M, mk_cal
CAL["pca"] = dict(n_components=N_PCA, variance_explained=expl)
print(f"PCA: {N_PCA} components explain {expl:.3f} of the variance")
json.dump(dict(CAL, variant=VARIANT, encoder=K2_INFO), open(f"{OUT}/calibration.json", "w"), indent=1)
tick("calibration")

## Embeddings for every cell (KRONOS2), and the KRONOS (v1) embeddings aligned to the same cells

In [ ]:
EMB = {}
for key, C in CO.items():
    Z = np.full((C["n"], D_EMB + N_PCA), np.nan, np.float16)
    idx = np.arange(C["n_img"])
    for s in range(0, len(idx), 50):
        E = embed_images(key, idx[s:s + 50], VARIANT)
        for i, (cl, mk) in E.items():
            a, b = C["starts"][i], C["starts"][i + 1]
            pc = ((torch.tensor(mk.astype(np.float32), device=dev) - PCA_MU) @ PCA_V).cpu().numpy()
            Z[a:b] = np.c_[cl, pc].astype(np.float16)
        del E
        tick(f"{key}: embedded {min(s + 50, len(idx))} / {len(idx)} images")
    ok = np.isfinite(Z.astype(np.float32)).all(1)
    print(f"{key}: cells with an embedding {ok.mean():.4f}")
    np.save(f"{OUT}/kronos2_emb_{key.lower()}.npy", Z)
    pd.DataFrame({"image_name": C["img_names"][np.repeat(np.arange(C["n_img"]), np.diff(C["starts"]))],
                  "object_number": C["obj"]}).to_parquet(f"{OUT}/kronos2_cells_{key.lower()}.parquet", index=False)
    EMB[key] = Z
    gc.collect()
tick("KRONOS2 embeddings saved")

# KRONOS (v1) embeddings from the kronos-zeroshot output, aligned to these cells by (image, object)
V1, D_V1 = {}, 384
for key, C in CO.items():
    cells1 = pd.read_parquet(g(f"kronos_cells_{key.lower()}.parquet")[0])
    pos = pd.Series(np.arange(len(cells1)), index=pd.MultiIndex.from_arrays([cells1.image_name.values, cells1.object_number.values]))
    names = C["img_names"][np.repeat(np.arange(C["n_img"]), np.diff(C["starts"]))]
    ix = pos.reindex(pd.MultiIndex.from_arrays([names, C["obj"]])).values
    assert not np.isnan(ix).any(), f"{key}: cells missing from the v1 embedding"
    ix = ix.astype(np.int64)
    if not SMOKE:
        assert (ix == np.arange(C["n"])).all(), f"{key}: v1 cell order differs"
    V1[key] = np.load(g(f"kronos_emb_{key.lower()}.npy")[0], mmap_mode="r")[ix]
    assert V1[key].shape[1] == D_V1 + N_PCA
    print(f"{key}: v1 embedding {V1[key].shape}, finite rows {np.isfinite(V1[key].astype(np.float32)).all(1).mean():.4f}")
tick("KRONOS (v1) embeddings aligned")

## Probes: cell typing, KRONOS2 and KRONOS vs the cell-table models (replicates 0-2)

In [ ]:
def k32_of(Z, d_cls):
    k = np.nan_to_num(Z[:, d_cls:d_cls + N_K].astype(np.float32))
    return (k - np.median(k, 0)) / (np.percentile(k, 95, 0) - np.percentile(k, 5, 0) + 1e-6)

ENC = {"KRONOS linear": "v1", "KRONOS2 linear": "v2"}
for key, C in CO.items():
    C["KF"] = {"v1": V1[key], "v2": EMB[key]}                                     # float16; rows are gathered when used
    k1, k2_ = k32_of(V1[key], D_V1), k32_of(EMB[key], D_EMB)
    C["XS"]["k32"] = np.c_[C["XS"]["cohort"], k1].astype(np.float32)
    C["XS"]["nbrk32"] = np.c_[C["XS"]["nbr"], k1].astype(np.float32)
    C["XS"]["k2_32"] = np.c_[C["XS"]["cohort"], k2_].astype(np.float32)
    C["XS"]["nbrk2_32"] = np.c_[C["XS"]["nbr"], k2_].astype(np.float32)
TABLE_ARMS = {"Table": "cohort", "Table + nbr": "nbr", "Table + K32": "k32", "Table + nbr + K32": "nbrk32",
              "Table + K2-32": "k2_32", "Table + nbr + K2-32": "nbrk2_32"}
FINE_ARMS = ("Table + nbr", "Table + nbr + K32", "Table + nbr + K2-32")
PAIRS = [(m, b) for m in list(TABLE_ARMS) + list(ENC) for b in ("Table", "Table + nbr") if m != b] + \
        [("KRONOS2 linear", "KRONOS linear"), ("Table + K2-32", "Table + K32"), ("Table + nbr + K2-32", "Table + nbr + K32")]

def combine(A, B):
    return dict(key="Both", n=A["n"] + B["n"], n_img=A["n_img"] + B["n_img"], starts=np.r_[A["starts"][:-1], B["starts"] + A["n"]],
                patient=np.r_[A["patient"], B["patient"]], cohort=np.r_[np.zeros(A["n"], np.int8), np.ones(B["n"], np.int8)],
                XS={k: np.r_[A["XS"][k], B["XS"][k]] for k in A["XS"]}, parts=(A, B),
                y={"coarse": np.r_[A["y"]["coarse"], B["y"]["coarse"]]})

def kf_rows(C, tag, rows):
    """Encoder embedding of the given rows as float32 (NaN -> 0, as in v1); pooled cohorts are read from their parts."""
    if C["key"] != "Both":
        return np.nan_to_num(C["KF"][tag][rows].astype(np.float32))
    A, B = C["parts"]
    out = np.empty((len(rows), A["KF"][tag].shape[1]), np.float32)
    ia = rows < A["n"]
    out[ia] = A["KF"][tag][rows[ia]]
    out[~ia] = B["KF"][tag][rows[~ia] - A["n"]]
    return np.nan_to_num(out)

def class_weights(y, ncls):
    c = np.bincount(y, minlength=ncls).astype(float)
    return np.where(c > 0, c.sum() / (ncls * np.maximum(c, 1)), 0.0)

def train_rows(C, task, imgs, seed):
    idx = cells_of(C, imgs)
    idx = idx[C["y"][task][idx] >= 0]
    rs = np.random.default_rng(seed)
    pools = [idx[C["cohort"][idx] == c] for c in (0, 1)] if C["key"] == "Both" else [idx]
    return np.sort(np.concatenate([rs.choice(p, min(N_FIT, len(p)), replace=False) for p in pools if len(p)]))

def fit_table(xkey, C, task, ncls, tr, seed):
    fit = train_rows(C, task, tr, seed)
    y = C["y"][task][fit]
    return lgb.LGBMClassifier(random_state=seed, **LGB).fit(C["XS"][xkey][fit], y, sample_weight=class_weights(y, ncls)[y])

def predict_rows(C, imgs, task):
    node = cells_of(C, imgs)
    return node[C["y"][task][node] >= 0]

def scorer(C, imgs, task, ncls, seed):
    node = predict_rows(C, imgs, task)
    pats, pc = np.unique(C["patient"][node], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
    y = C["y"][task][node]
    return dict(node=node, y=y, pc=pc, npat=len(pats), W=W, ncls=ncls, present=np.bincount(y, minlength=ncls) > 0)

def metrics(Mx, ncls, present):
    Mx = Mx.reshape(-1, ncls, ncls)
    tp = np.einsum("bii->bi", Mx)
    sup, prd = Mx.sum(2), Mx.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where(sup + prd > 0, 2 * tp / (sup + prd), 0.0)
        rec = np.where(sup > 0, tp / sup, np.nan)
        out = dict(macro_f1=f1[:, present].mean(1), bal_acc=np.nanmean(rec[:, present], 1), acc=tp.sum(1) / Mx.sum((1, 2)))
    return out, f1

def score(S, pred_full):
    p = pred_full[S["node"]]
    assert (p >= 0).all(), "missing predictions"
    k = S["ncls"]
    cm = np.bincount((S["pc"] * k + S["y"]) * k + p, minlength=S["npat"] * k * k).reshape(S["npat"], k * k).astype(np.float64)
    pt, f1pt = metrics(cm.sum(0, keepdims=True), k, S["present"])
    bt, _ = metrics(S["W"] @ cm, k, S["present"])
    return {m: float(v[0]) for m, v in pt.items()}, bt, f1pt[0]

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

UNITS = [("coarse", "Jackson"), ("coarse", "Danenberg"), ("coarse", "Both"), ("fine", "Jackson"), ("fine", "Danenberg")]
ROWS, DIFF, PCL = [], [], []
for REP in REPS:
    IMG = IMG0 if REP == 0 else make_splits(REP)
    SCR = {(task, t): scorer(CO[t], IMG[t]["test"], task, len(COARSE) if task == "coarse" else len(CO[t]["fine"]),
                             SEED + 31 + REP + 1000 * (t == "Danenberg"))
           for task in ("coarse", "fine") for t in ("Jackson", "Danenberg")}
    BOOT = {}
    for task, src in UNITS:
        if (time.time() - T0) / 60 > BUDGET_MIN:
            FAILED.append(f"budget: rep {REP} {task} {src}")
            continue
        C = combine(CJ, CD) if src == "Both" else CO[src]
        ncls = len(COARSE) if task == "coarse" else len(CO[src]["fine"])
        tests = ["Jackson", "Danenberg"] if task == "coarse" else [src]
        try:
            preds = {}
            arms = TABLE_ARMS if task == "coarse" else {k_: TABLE_ARMS[k_] for k_ in FINE_ARMS}
            for name, xkey in arms.items():
                mdl = fit_table(xkey, C, task, ncls, IMG[src]["train"], SEED + REP)
                for t in tests:
                    pr = np.full(CO[t]["n"], -1, np.int64)
                    node = predict_rows(CO[t], IMG[t]["test"], task)
                    pr[node] = mdl.classes_[mdl.predict_proba(CO[t]["XS"][xkey][node]).argmax(1)]
                    preds[(name, t)] = pr
            fit = train_rows(C, task, IMG[src]["train"], SEED + REP)
            vrows = np.concatenate([predict_rows(CO[s_], IMG[s_]["val"], task) + (CJ["n"] if src == "Both" and s_ == "Danenberg" else 0)
                                    for s_ in (["Jackson", "Danenberg"] if src == "Both" else [src])])
            for name, tag in ENC.items():
                Zf = kf_rows(C, tag, fit)
                sc = scale_fit(Zf)
                f1v, wd, lin = fit_logistic(sc(Zf), C["y"][task][fit], ncls, [1e-4, 1e-3, 1e-2],
                                            sc(kf_rows(C, tag, vrows)), C["y"][task][vrows], SEED + REP)
                del Zf
                for t in tests:
                    pr = np.full(CO[t]["n"], -1, np.int64)
                    node = predict_rows(CO[t], IMG[t]["test"], task)
                    with torch.no_grad():
                        pr[node] = lin(torch.tensor(sc(kf_rows(CO[t], tag, node)), device=dev)).argmax(1).cpu().numpy()
                    preds[(name, t)] = pr
                del lin
                torch.cuda.empty_cache()
            for (name, t), pr in preds.items():
                pt, bt, f1c = score(SCR[(task, t)], pr)
                BOOT[(task, src, t, name)] = bt
                for m_, v in pt.items():
                    lo, hi = ci(bt[m_])
                    ROWS.append(dict(rep=REP, task=task, train=src, test=t, model=name, metric=m_, value=v, lo=lo, hi=hi))
                labs = COARSE if task == "coarse" else CO[t]["fine"]
                for c_ in np.flatnonzero(SCR[(task, t)]["present"]):
                    PCL.append(dict(rep=REP, task=task, train=src, test=t, model=name, label=labs[c_], f1=float(f1c[c_])))
            tick(f"rep {REP} {task} / train {src}: " + ", ".join(
                f"{n_}@{t[0]} {score(SCR[(task, t)], p_)[0]['macro_f1']:.3f}" for (n_, t), p_ in preds.items()))
            if src == "Both":
                del C
                gc.collect()
        except Exception:
            print(f"!!! rep {REP} {task} {src} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(f"rep {REP} {task} {src}")
    R_ = pd.DataFrame(ROWS)
    R_ = R_[(R_.rep == REP) & (R_.metric == "macro_f1")]
    for (task, src, t, name), bt in BOOT.items():
        for m_, base in PAIRS:
            kb = (task, src, t, base)
            if m_ != name or kb not in BOOT:
                continue
            d = R_[(R_.task == task) & (R_.train == src) & (R_.test == t)].set_index("model").value
            lo, hi = ci(bt["macro_f1"] - BOOT[kb]["macro_f1"])
            DIFF.append(dict(rep=REP, task=task, train=src, test=t, model=name, base=base, diff=float(d[name] - d[base]),
                             lo=lo, hi=hi, call="better" if lo > 0 else "worse" if hi < 0 else "n.s."))
RES, DIFFD = pd.DataFrame(ROWS), pd.DataFrame(DIFF)
RES.to_csv(f"{OUT}/scores.csv", index=False)
DIFFD.to_csv(f"{OUT}/diffs.csv", index=False)
pd.DataFrame(PCL).to_csv(f"{OUT}/perclass.csv", index=False)
with pd.option_context("display.width", 300, "display.max_columns", 20):
    f1t = RES[RES.metric == "macro_f1"]
    print(f1t.pivot_table(index=["rep", "task", "train", "test"], columns="model", values="value").round(3).to_string())
    print("\nmean (SD) over replicates")
    agg = f1t.groupby(["task", "train", "test", "model"]).value.agg(["mean", "std"]).round(3)
    print(agg.unstack("model").to_string())
    print(DIFFD[DIFFD.base.isin(["Table + nbr", "KRONOS linear", "Table + nbr + K32", "Table + K32"])].round(3).to_string(index=False))

## Gate for adaptation (both encoders) and the head-to-head

In [ ]:
GATE_ARMS = {"KRONOS": ("KRONOS linear", "Table + nbr + K32"), "KRONOS2": ("KRONOS2 linear", "Table + nbr + K2-32")}
f1 = RES[RES.metric == "macro_f1"]
reps_run = sorted(f1.rep.unique().tolist())
gate = {}
for enc, (lin_arm, k_arm) in GATE_ARMS.items():
    per_rep = {}
    for REP in reps_run:
        a = b = False
        det = []
        fr = f1[f1.rep == REP]
        for src in ("Jackson", "Danenberg"):
            s = fr[(fr.task == "coarse") & (fr.train == src) & (fr.test == src)].set_index("model").value
            if lin_arm in s and "Table + nbr" in s:
                det.append(f"(a) {src[0]}->{src[0]}: {lin_arm} {s[lin_arm]:.3f} vs Table + nbr {s['Table + nbr']:.3f} "
                           f"({s[lin_arm] - s['Table + nbr']:+.3f})")
                a |= bool(s[lin_arm] >= s["Table + nbr"] - 0.05)
        bb = DIFFD[(DIFFD.rep == REP) & (DIFFD.task == "coarse") & (DIFFD.model == k_arm) & (DIFFD.base == "Table + nbr")]
        for r in bb.itertuples():
            det.append(f"(b) {r.train[0]}->{r.test[0]}: {k_arm} {r.diff:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}]")
            b |= bool(r.call == "better" and r.diff >= 0.01)
        per_rep[int(REP)] = dict(a=a, b=b, passed=a or b, details=det)
    gate[enc] = dict(per_replicate=per_rep,
                     rep0_rule_v1=per_rep.get(0, {}).get("passed"),
                     passes_every_replicate=bool(len(per_rep) == len(REPS) and all(v["passed"] for v in per_rep.values())),
                     replicates_run=reps_run)
    gate[enc]["proceed_to_adaptation"] = gate[enc]["passes_every_replicate"]

h2h = DIFFD[(DIFFD.task == "coarse") & (((DIFFD.model == "KRONOS2 linear") & (DIFFD.base == "KRONOS linear")) |
                                       ((DIFFD.model == "Table + nbr + K2-32") & (DIFFD.base == "Table + nbr + K32")))]
H2H = h2h.groupby(["model", "train", "test"]).agg(mean_diff=("diff", "mean"), min_diff=("diff", "min"), max_diff=("diff", "max"),
                                                  n_better=("call", lambda c: int((c == "better").sum())),
                                                  n_worse=("call", lambda c: int((c == "worse").sum())), n=("call", "size")).round(4)
H2H.to_csv(f"{OUT}/head_to_head.csv")
print(H2H.to_string())
json.dump(dict(gate=gate, calibration=CAL, variant=VARIANT, reps=REPS, failed=FAILED, smoke=SMOKE,
               minutes=(time.time() - T0) / 60, kronos2=K2_INFO), open(f"{OUT}/summary.json", "w"), indent=1)
print(json.dumps(gate, indent=1))
print("failed blocks:", FAILED if FAILED else "none")
tick("done")